# Lab 9: Single-Path Chunking and First LLM Answer
## One Shared Corpus, One Chunk Table, One Grounded Generation Flow

This lab turns the RAG pipeline into one clean end-to-end workflow.

In Lab 8, you already learned retrieval, context building, and prompt structure.
In this lab, we keep the design simpler:

- every document has one `text` field,
- we split that text into retrievable chunks,
- we retrieve chunk-level evidence,
- the LLM appears only at the end as the final answer generator.

The central idea of this notebook is:

> The LLM answers from what you give it. Chunking decides what evidence it gets to see.


## The Big Picture

```text
domain documents
-> chunking
-> retrieval
-> context package
-> final LLM answer
```

We will work with three mini-domains:

- healthcare
- e-commerce
- software documentation

This lab keeps the same chunking style you already saw in Lab 8 so you can focus on the end-to-end RAG flow.


## Learning Outcomes

By the end of this lab, you should be able to explain and implement:

| Topic | What you must understand |
|---|---|
| Chunking | How fixed overlapping chunks support retrieval before generation |
| Domain variation | Why healthcare, e-commerce, and software docs still need different evidence inspection habits |
| Retrieval | How TF-IDF, lightweight semantic approximation, and hybrid retrieval rank chunks |
| Metadata-aware filtering | Why current/outdated, platform, region, and audience still matter |
| Context packaging | How to turn candidate chunks into final evidence for generation |
| First LLM connection | How to connect an LLM only after retrieval and context are ready |
| Failure analysis | How to separate chunking, retrieval, context, and generation failures |


## Section Map

1. Install and import libraries
2. Build a shared mixed-domain corpus
3. Build one shared chunk table with fixed chunking
4. Retrieve relevant chunks
5. Evaluate retrieval quality
6. Build final context packages
7. Connect the first LLM
8. Analyze failures
9. Complete the final assignment


# Section 1  Install Required Packages

Run this cell once if these packages are missing in your environment.

We keep the stack intentionally light:

- `pandas` and `numpy` for tables and calculations
- `scikit-learn` for lexical and lightweight semantic retrieval
- `requests` for the final local Ollama call


In [24]:
!pip install -q pandas numpy scikit-learn requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ahmed\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Section 2 — Import Libraries


In [25]:
import os
import re
from textwrap import shorten

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)

# Section 3 — Why Chunking Matters

Chunking changes what is retrievable and what is answerable.

In this lab, we keep the same overlapping word-window style from Lab 8.
That way, the new learning focus is the full RAG pipeline rather than a new chunking method.

Typical failure patterns:

- a rule is separated from its exception,
- a step-by-step procedure is split across unrelated chunks,
- a troubleshooting symptom is separated from its fix,
- an outdated notice looks relevant because the words match.


In [26]:
examples = pd.DataFrame(
    [
        {
            "failure_type": "Rule / exception split",
            "bad_chunking_example": "Return policy in one chunk, opened-electronics exception in another chunk",
            "downstream_risk": "Model answers with the general rule and misses the exception",
        },
        {
            "failure_type": "Procedure split",
            "bad_chunking_example": "Steps 1-2 in one chunk, critical step 3 in a different chunk",
            "downstream_risk": "Answer becomes incomplete or out of order",
        },
        {
            "failure_type": "Symptom / fix split",
            "bad_chunking_example": "Windows login loop described in one chunk, cookie reset fix in another",
            "downstream_risk": "Retriever finds the symptom but not the resolution",
        },
        {
            "failure_type": "Current / outdated conflict",
            "bad_chunking_example": "Old fasting rule retrieved beside the current preparation guide",
            "downstream_risk": "Answer may be textually relevant but unsafe",
        },
    ]
)

examples

,failure_type,bad_chunking_example,downstream_risk
0,Rule / exception split,"Return policy in one chunk, opened-electronics exception in another chunk",Model answers with the general rule and misses the exception
1,Procedure split,"Steps 1-2 in one chunk, critical step 3 in a different chunk",Answer becomes incomplete or out of order
2,Symptom / fix split,"Windows login loop described in one chunk, cookie reset fix in another",Retriever finds the symptom but not the resolution
3,Current / outdated conflict,Old fasting rule retrieved beside the current preparation guide,Answer may be textually relevant but unsafe


# Section 4 — Build the Shared Multi-Domain Corpus

We use a shared, instructor-provided corpus so that every student evaluates the same failure cases.

The corpus has three mini-domains:

- healthcare
- e-commerce
- software docs

Every document contains:

- metadata
- one `text` field that will be split with the same fixed overlapping chunking style used in Lab 8


In [27]:
documents = [{'domain': 'healthcare',
  'document_id': 0,
  'title': 'Urgent Care Triage Guide',
  'doc_type': 'triage',
  'effective_date': '2026-02-15',
  'is_current': True,
  'audience': 'adult',
  'urgency_level': 'mixed',
  'region': None,
  'platform': None,
  'version': None,
  'text': 'Emergency signs: chest pain with breathing difficulty, sudden confusion, blue lips, or '
          'loss of consciousness require emergency services immediately.\n'
          '\n'
          'Same-day urgent clinic care is appropriate for fever above 39C, worsening asthma '
          'symptoms, dehydration, or severe vomiting that is still manageable without ambulance '
          'transfer.\n'
          '\n'
          'Mild cold symptoms, routine refills, and paperwork requests should be booked as '
          'standard appointments rather than urgent walk-in care.'},
 {'domain': 'healthcare',
  'document_id': 1,
  'title': 'Fasting Blood Test Preparation',
  'doc_type': 'preparation guide',
  'effective_date': '2026-03-10',
  'is_current': True,
  'audience': 'adult',
  'urgency_level': 'routine',
  'region': None,
  'platform': None,
  'version': None,
  'text': 'For a fasting blood test, stop eating 8 hours before your appointment. Plain water is '
          'allowed and is encouraged unless your clinician gave a different instruction.\n'
          '\n'
          'Avoid coffee, tea, juice, and sports drinks during the fasting window because they can '
          'change glucose and lipid readings.\n'
          '\n'
          'Take regular medicines only if your clinician told you they are safe before the test. '
          'Bring your medication list to the appointment.'},
 {'domain': 'healthcare',
  'document_id': 2,
  'title': 'Antibiotic Course Instructions',
  'doc_type': 'medication guide',
  'effective_date': '2026-01-05',
  'is_current': True,
  'audience': 'adult',
  'urgency_level': 'routine',
  'region': None,
  'platform': None,
  'version': None,
  'text': 'Finish the full antibiotic course even if symptoms improve after the first few doses. '
          'Stopping early increases the risk of relapse and resistant bacteria.\n'
          '\n'
          'If a dose is missed, take it when remembered unless the next dose is due soon. Do not '
          'double the dose.\n'
          '\n'
          'Seek urgent review if rash, facial swelling, wheezing, or severe diarrhea develops '
          'while taking the medicine.'},
 {'domain': 'healthcare',
  'document_id': 3,
  'title': 'Pediatric Appointment Rules',
  'doc_type': 'policy',
  'effective_date': '2026-02-01',
  'is_current': True,
  'audience': 'pediatric',
  'urgency_level': 'routine',
  'region': None,
  'platform': None,
  'version': None,
  'text': 'Children under 16 must attend with a parent, guardian, or documented caregiver unless '
          'emergency stabilization is already in progress.\n'
          '\n'
          'Routine pediatric visits require an appointment. Walk-ins are accepted only for urgent '
          'breathing issues, uncontrolled fever, or injury.\n'
          '\n'
          'Bring the vaccination record and current medication list if the visit is for follow-up '
          'or school documentation.'},
 {'domain': 'healthcare',
  'document_id': 4,
  'title': 'Old Fasting Advisory',
  'doc_type': 'old notice',
  'effective_date': '2024-09-01',
  'is_current': False,
  'audience': 'adult',
  'urgency_level': 'routine',
  'region': None,
  'platform': None,
  'version': None,
  'text': 'Legacy fasting advice from 2024 stated that patients should avoid both food and water '
          'after midnight before morning blood work.\n'
          '\n'
          'This notice remains archived for audit reasons and should not replace the current '
          'fasting blood test preparation guide.'},
 {'domain': 'ecommerce',
  'document_id': 5,
  'title': 'Returns and Refunds Policy',
  'doc_type': 'policy',
  'effective_date': '2026-02-20',
  'is_current': True,
  'audience': 'consumer',
  'urgency_level': None,
  'region': 'global',
  'platform': None,
  'version': None,
  'text': 'Most unopened products can be returned within 30 days of delivery for a full refund if '
          'they include all original accessories and packaging.\n'
          '\n'
          'Opened hygiene products, activated software keys, and opened power accessories for '
          'laptops are not eligible for standard return unless they arrived damaged or defective.\n'
          '\n'
          'Refunds are issued to the original payment method after inspection, usually within 5 '
          'business days of warehouse approval.'},
 {'domain': 'ecommerce',
  'document_id': 6,
  'title': 'Damaged Delivery Support',
  'doc_type': 'support procedure',
  'effective_date': '2026-03-12',
  'is_current': True,
  'audience': 'consumer',
  'urgency_level': None,
  'region': 'global',
  'platform': None,
  'version': None,
  'text': 'If an order arrives damaged, photograph the item, the shipping box, and the label '
          'before discarding the packaging.\n'
          '\n'
          'Submit the damage report through the order page within 48 hours. Use the reason code '
          "'damaged on arrival' and attach the photos.\n"
          '\n'
          'Do not mail the product back until support confirms whether a pickup, replacement, or '
          'disposal authorization applies.'},
 {'domain': 'ecommerce',
  'document_id': 7,
  'title': 'Shipping Times FAQ',
  'doc_type': 'FAQ',
  'effective_date': '2026-01-18',
  'is_current': True,
  'audience': 'consumer',
  'urgency_level': None,
  'region': 'UAE',
  'platform': None,
  'version': None,
  'text': 'Standard delivery in the UAE usually takes 2 to 4 business days after dispatch. Remote '
          'areas may require one extra business day.\n'
          '\n'
          'Express delivery in the UAE usually arrives the next business day if the order is '
          'placed before 3 PM Gulf Standard Time.\n'
          '\n'
          'Heavy items and pre-orders follow separate timelines shown at checkout and should not '
          'be estimated from the standard FAQ alone.'},
 {'domain': 'ecommerce',
  'document_id': 8,
  'title': 'Laptop Warranty Guide',
  'doc_type': 'warranty',
  'effective_date': '2026-02-28',
  'is_current': True,
  'audience': 'consumer',
  'urgency_level': None,
  'region': 'global',
  'platform': None,
  'version': None,
  'text': 'New laptops include a 24-month limited hardware warranty covering manufacturing defects '
          'in the main unit and internal battery.\n'
          '\n'
          'Chargers, sleeves, and promotional accessories are covered for 12 months unless a '
          'product page states a shorter promotional warranty.\n'
          '\n'
          'Accidental damage, liquid exposure, and unauthorized repair are excluded from warranty '
          'coverage.'},
 {'domain': 'ecommerce',
  'document_id': 9,
  'title': 'Old Holiday Return Extension',
  'doc_type': 'old notice',
  'effective_date': '2024-12-01',
  'is_current': False,
  'audience': 'consumer',
  'urgency_level': None,
  'region': 'global',
  'platform': None,
  'version': None,
  'text': 'During the 2024 holiday campaign, selected unopened electronics could be returned '
          'within 45 days instead of the normal 30-day policy window.\n'
          '\n'
          'This notice applied only to the seasonal campaign and should not override the current '
          'returns and refunds policy.'},
 {'domain': 'software_docs',
  'document_id': 10,
  'title': 'Desktop App 5.2 Installation Guide',
  'doc_type': 'installation guide',
  'effective_date': '2026-03-01',
  'is_current': True,
  'audience': 'developer',
  'urgency_level': None,
  'region': None,
  'platform': 'Windows, macOS',
  'version': '5.2',
  'text': 'Desktop App 5.2 requires Python 3.11, Node.js 20, and administrator rights during the '
          'local service installation step.\n'
          '\n'
          'On Windows, run the installer from an elevated PowerShell window so the local '
          'background service can register correctly.\n'
          '\n'
          'On macOS, approve the background helper in System Settings after the first launch if '
          'the helper is blocked.'},
 {'domain': 'software_docs',
  'document_id': 11,
  'title': 'Login Loop Troubleshooting',
  'doc_type': 'troubleshooting',
  'effective_date': '2026-03-14',
  'is_current': True,
  'audience': 'developer',
  'urgency_level': None,
  'region': None,
  'platform': 'Windows, macOS',
  'version': '5.2',
  'text': 'If the app returns to the login screen immediately after authentication, first confirm '
          'that the system clock is correct and the browser can store secure cookies.\n'
          '\n'
          'On Windows, corporate endpoint tools may block loopback redirects. Add the desktop '
          'helper to the local allow-list before retrying login.\n'
          '\n'
          'If the browser opens but the token never returns to the app, clear cached cookies for '
          'the app domain and retry with the latest desktop helper.'},
 {'domain': 'software_docs',
  'document_id': 12,
  'title': 'Storage Backend Configuration',
  'doc_type': 'configuration reference',
  'effective_date': '2026-02-07',
  'is_current': True,
  'audience': 'developer',
  'urgency_level': None,
  'region': None,
  'platform': 'cross-platform',
  'version': '5.2',
  'text': 'To switch from the local SQLite backend to Postgres, set STORAGE_BACKEND=postgres and '
          'provide APP_DB_HOST, APP_DB_PORT, APP_DB_NAME, APP_DB_USER, and APP_DB_PASSWORD.\n'
          '\n'
          'After changing the backend, run appctl migrate before the next start so schema changes '
          'are applied to the target database.\n'
          '\n'
          'If SSL is required by the database provider, set APP_DB_SSLMODE=require or the '
          'connection test will fail even if the credentials are correct.'},
 {'domain': 'software_docs',
  'document_id': 13,
  'title': 'Migration Notes from 4.x to 5.2',
  'doc_type': 'migration guide',
  'effective_date': '2026-02-18',
  'is_current': True,
  'audience': 'developer',
  'urgency_level': None,
  'region': None,
  'platform': 'cross-platform',
  'version': '5.2',
  'text': 'Version 5.2 removed the legacy embedded browser callback used in the 4.x desktop client '
          'and now requires the desktop helper for secure token return.\n'
          '\n'
          'The config file path changed from config.yaml to appsettings.toml. Existing 4.x '
          'deployments should export settings before upgrade.\n'
          '\n'
          'Custom plugins built for 4.x must be recompiled against the 5.2 SDK before deployment.'},
 {'domain': 'software_docs',
  'document_id': 14,
  'title': 'Deprecated 4.x Setup Guide',
  'doc_type': 'old notice',
  'effective_date': '2024-11-30',
  'is_current': False,
  'audience': 'developer',
  'urgency_level': None,
  'region': None,
  'platform': 'Windows, macOS',
  'version': '4.x',
  'text': 'The 4.x setup guide instructs users to complete authentication through the embedded '
          'browser callback and to store settings in config.yaml.\n'
          '\n'
          'This guide is archived for legacy reference and should not be used for 5.2 '
          'deployments.'}]

documents_df = pd.DataFrame(documents)

documents_df[["domain", "document_id", "title", "doc_type", "effective_date", "is_current"]]

,domain,document_id,title,doc_type,effective_date,is_current
0,healthcare,0,Urgent Care Triage Guide,triage,2026-02-15,True
1,healthcare,1,Fasting Blood Test Preparation,preparation guide,2026-03-10,True
2,healthcare,2,Antibiotic Course Instructions,medication guide,2026-01-05,True
3,healthcare,3,Pediatric Appointment Rules,policy,2026-02-01,True
4,healthcare,4,Old Fasting Advisory,old notice,2024-09-01,False
5,ecommerce,5,Returns and Refunds Policy,policy,2026-02-20,True
6,ecommerce,6,Damaged Delivery Support,support procedure,2026-03-12,True
7,ecommerce,7,Shipping Times FAQ,FAQ,2026-01-18,True
8,ecommerce,8,Laptop Warranty Guide,warranty,2026-02-28,True
9,ecommerce,9,Old Holiday Return Extension,old notice,2024-12-01,False


In [28]:
domain_summary = (
    documents_df.groupby(["domain", "doc_type"], as_index=False)
    .size()
    .rename(columns={"size": "document_count"})
    .sort_values(["domain", "document_count"], ascending=[True, False])
)

domain_summary

,domain,doc_type,document_count
0,ecommerce,FAQ,1
1,ecommerce,old notice,1
2,ecommerce,policy,1
3,ecommerce,support procedure,1
4,ecommerce,warranty,1
5,healthcare,medication guide,1
6,healthcare,old notice,1
7,healthcare,policy,1
8,healthcare,preparation guide,1
9,healthcare,triage,1


## Query Set and Ground Truth

Each query is labeled with:

- domain
- query type
- user profile
- optional metadata hints such as platform, version, region, or audience
- relevant document titles


In [29]:
queries = [
    {
        "query_id": "H1",
        "domain": "healthcare",
        "query": "Can I drink water before a fasting blood test?",
        "query_type": "factual",
        "user_profile": "routine patient",
        "platform": None,
        "version": None,
        "region": None,
        "audience": "adult",
        "relevant_titles": ["Fasting Blood Test Preparation"],
    },
    {
        "query_id": "H2",
        "domain": "healthcare",
        "query": "What should I do if I have chest pain and trouble breathing?",
        "query_type": "troubleshooting",
        "user_profile": "urgent patient",
        "platform": None,
        "version": None,
        "region": None,
        "audience": "adult",
        "relevant_titles": ["Urgent Care Triage Guide"],
    },
    {
        "query_id": "H3",
        "domain": "healthcare",
        "query": "Do I have to finish the antibiotic course if I feel better after two days?",
        "query_type": "policy",
        "user_profile": "routine patient",
        "platform": None,
        "version": None,
        "region": None,
        "audience": "adult",
        "relevant_titles": ["Antibiotic Course Instructions"],
    },
    {
        "query_id": "H4",
        "domain": "healthcare",
        "query": "Can a 12-year-old come alone to a clinic appointment?",
        "query_type": "policy",
        "user_profile": "parent",
        "platform": None,
        "version": None,
        "region": None,
        "audience": "pediatric",
        "relevant_titles": ["Pediatric Appointment Rules"],
    },
    {
        "query_id": "E1",
        "domain": "ecommerce",
        "query": "Can I return an opened laptop charger?",
        "query_type": "policy",
        "user_profile": "buyer",
        "platform": None,
        "version": None,
        "region": "global",
        "audience": "consumer",
        "relevant_titles": ["Returns and Refunds Policy"],
    },
    {
        "query_id": "E2",
        "domain": "ecommerce",
        "query": "My order arrived with a cracked screen. What should I do first?",
        "query_type": "troubleshooting",
        "user_profile": "buyer",
        "platform": None,
        "version": None,
        "region": "global",
        "audience": "consumer",
        "relevant_titles": ["Damaged Delivery Support"],
    },
    {
        "query_id": "E3",
        "domain": "ecommerce",
        "query": "How long does normal shipping take in the UAE?",
        "query_type": "factual",
        "user_profile": "buyer",
        "platform": None,
        "version": None,
        "region": "UAE",
        "audience": "consumer",
        "relevant_titles": ["Shipping Times FAQ"],
    },
    {
        "query_id": "E4",
        "domain": "ecommerce",
        "query": "Does the laptop battery come with a two-year warranty?",
        "query_type": "factual",
        "user_profile": "buyer",
        "platform": None,
        "version": None,
        "region": "global",
        "audience": "consumer",
        "relevant_titles": ["Laptop Warranty Guide"],
    },
    {
        "query_id": "S1",
        "domain": "software_docs",
        "query": "Which Python version is required for Desktop App 5.2?",
        "query_type": "factual",
        "user_profile": "developer",
        "platform": None,
        "version": "5.2",
        "region": None,
        "audience": "developer",
        "relevant_titles": ["Desktop App 5.2 Installation Guide"],
    },
    {
        "query_id": "S2",
        "domain": "software_docs",
        "query": "The app keeps sending me back to login on Windows. What should I check first?",
        "query_type": "troubleshooting",
        "user_profile": "developer",
        "platform": "Windows",
        "version": "5.2",
        "region": None,
        "audience": "developer",
        "relevant_titles": ["Login Loop Troubleshooting"],
    },
    {
        "query_id": "S3",
        "domain": "software_docs",
        "query": "How do I switch the storage backend to Postgres?",
        "query_type": "procedure",
        "user_profile": "developer",
        "platform": None,
        "version": "5.2",
        "region": None,
        "audience": "developer",
        "relevant_titles": ["Storage Backend Configuration"],
    },
    {
        "query_id": "S4",
        "domain": "software_docs",
        "query": "Can I still use the old 4.x setup steps for a 5.2 deployment?",
        "query_type": "policy",
        "user_profile": "developer",
        "platform": None,
        "version": "5.2",
        "region": None,
        "audience": "developer",
        "relevant_titles": ["Migration Notes from 4.x to 5.2"],
    },
]

queries_df = pd.DataFrame(queries)

title_to_id = dict(zip(documents_df["title"], documents_df["document_id"]))
queries_df["relevant_document_ids"] = queries_df["relevant_titles"].apply(
    lambda titles: [title_to_id[title] for title in titles]
)

queries_df[["query_id", "domain", "query_type", "query", "relevant_titles"]]

,query_id,domain,query_type,query,relevant_titles
0,H1,healthcare,factual,Can I drink water before a fasting blood test?,[Fasting Blood Test Preparation]
1,H2,healthcare,troubleshooting,What should I do if I have chest pain and trouble breathing?,[Urgent Care Triage Guide]
2,H3,healthcare,policy,Do I have to finish the antibiotic course if I feel better after two days?,[Antibiotic Course Instructions]
3,H4,healthcare,policy,Can a 12-year-old come alone to a clinic appointment?,[Pediatric Appointment Rules]
4,E1,ecommerce,policy,Can I return an opened laptop charger?,[Returns and Refunds Policy]
5,E2,ecommerce,troubleshooting,My order arrived with a cracked screen. What should I do first?,[Damaged Delivery Support]
6,E3,ecommerce,factual,How long does normal shipping take in the UAE?,[Shipping Times FAQ]
7,E4,ecommerce,factual,Does the laptop battery come with a two-year warranty?,[Laptop Warranty Guide]
8,S1,software_docs,factual,Which Python version is required for Desktop App 5.2?,[Desktop App 5.2 Installation Guide]
9,S2,software_docs,troubleshooting,The app keeps sending me back to login on Windows. What should I check first?,[Login Loop Troubleshooting]


# Section 5 — Build One Shared Chunk Table

We use one chunking path in this lab.

The goal is simple:

- start from one document `text` field,
- split it into sentences,
- group nearby sentences into bounded chunks,
- retrieve those chunks as focused evidence.


In [30]:
def chunk_text(text, chunk_size=38, overlap=10):
    words = text.split()

    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0:
        raise ValueError("overlap cannot be negative")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
        start += chunk_size - overlap
    return chunks


def build_chunks(documents_frame, chunk_size=38, overlap=10):
    rows = []

    for _, doc in documents_frame.iterrows():
        chunks = chunk_text(
            doc["text"],
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_index, chunk_text_value in enumerate(chunks):
            rows.append(
                {
                    "chunk_id": f"{doc['domain']}_doc{doc['document_id']}_chunk_{chunk_index}",
                    "domain": doc["domain"],
                    "document_id": doc["document_id"],
                    "title": doc["title"],
                    "doc_type": doc["doc_type"],
                    "effective_date": doc["effective_date"],
                    "is_current": doc["is_current"],
                    "audience": doc["audience"],
                    "urgency_level": doc["urgency_level"],
                    "region": doc["region"],
                    "platform": doc["platform"],
                    "version": doc["version"],
                    "chunk_index": chunk_index,
                    "chunk_text": chunk_text_value,
                    "word_count": len(chunk_text_value.split()),
                    "search_text": " ".join(
                        [
                            str(doc["title"]),
                            str(doc["doc_type"]),
                            str(doc["domain"]),
                            str(doc["audience"]),
                            str(doc["region"]),
                            str(doc["platform"]),
                            str(doc["version"]),
                            chunk_text_value,
                        ]
                    ),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks(documents_df)
chunks_df.head(10)


,chunk_id,domain,document_id,title,doc_type,effective_date,is_current,audience,urgency_level,region,platform,version,chunk_index,chunk_text,word_count,search_text
0,healthcare_doc0_chunk_0,healthcare,0,Urgent Care Triage Guide,triage,2026-02-15,True,adult,mixed,None,None,None,0,"Emergency signs: chest pain with breathing difficulty, sudden confusion, blue lips, or loss of consciousness require emergency services immediately. Same-day urgent clinic care...",38,"Urgent Care Triage Guide triage healthcare adult None None None Emergency signs: chest pain with breathing difficulty, sudden confusion, blue lips, or loss of consciousness req..."
1,healthcare_doc0_chunk_1,healthcare,0,Urgent Care Triage Guide,triage,2026-02-15,True,adult,mixed,None,None,None,1,"39C, worsening asthma symptoms, dehydration, or severe vomiting that is still manageable without ambulance transfer. Mild cold symptoms, routine refills, and paperwork requests...",34,"Urgent Care Triage Guide triage healthcare adult None None None 39C, worsening asthma symptoms, dehydration, or severe vomiting that is still manageable without ambulance trans..."
2,healthcare_doc1_chunk_0,healthcare,1,Fasting Blood Test Preparation,preparation guide,2026-03-10,True,adult,routine,None,None,None,0,"For a fasting blood test, stop eating 8 hours before your appointment. Plain water is allowed and is encouraged unless your clinician gave a different instruction. Avoid coffee...",38,"Fasting Blood Test Preparation preparation guide healthcare adult None None None For a fasting blood test, stop eating 8 hours before your appointment. Plain water is allowed a..."
3,healthcare_doc1_chunk_1,healthcare,1,Fasting Blood Test Preparation,preparation guide,2026-03-10,True,adult,routine,None,None,None,1,"tea, juice, and sports drinks during the fasting window because they can change glucose and lipid readings. Take regular medicines only if your clinician told you they are safe...",38,"Fasting Blood Test Preparation preparation guide healthcare adult None None None tea, juice, and sports drinks during the fasting window because they can change glucose and lip..."
4,healthcare_doc1_chunk_2,healthcare,1,Fasting Blood Test Preparation,preparation guide,2026-03-10,True,adult,routine,None,None,None,2,safe before the test. Bring your medication list to the appointment.,11,Fasting Blood Test Preparation preparation guide healthcare adult None None None safe before the test. Bring your medication list to the appointment.
5,healthcare_doc2_chunk_0,healthcare,2,Antibiotic Course Instructions,medication guide,2026-01-05,True,adult,routine,None,None,None,0,"Finish the full antibiotic course even if symptoms improve after the first few doses. Stopping early increases the risk of relapse and resistant bacteria. If a dose is missed, ...",38,Antibiotic Course Instructions medication guide healthcare adult None None None Finish the full antibiotic course even if symptoms improve after the first few doses. Stopping e...
6,healthcare_doc2_chunk_1,healthcare,2,Antibiotic Course Instructions,medication guide,2026-01-05,True,adult,routine,None,None,None,1,"missed, take it when remembered unless the next dose is due soon. Do not double the dose. Seek urgent review if rash, facial swelling, wheezing, or severe diarrhea develops whi...",33,"Antibiotic Course Instructions medication guide healthcare adult None None None missed, take it when remembered unless the next dose is due soon. Do not double the dose. Seek u..."
7,healthcare_doc3_chunk_0,healthcare,3,Pediatric Appointment Rules,policy,2026-02-01,True,pediatric,routine,None,None,None,0,"Children under 16 must attend with a parent, guardian, or documented caregiver unless emergency stabilization is already in progress. Routine pediatric visits require an appoin...",38,"Pediatric Appointment Rules policy healthcare pediatric None None None Children under 16 must attend with a parent, guardian, or documented caregiver unless emergency stabiliza..."
8,healthcare_doc3_

In [31]:
chunk_summary = (
    chunks_df.groupby(["domain"], as_index=False)
    .agg(
        chunk_count=("chunk_id", "count"),
        avg_words=("word_count", "mean"),
        min_words=("word_count", "min"),
        max_words=("word_count", "max"),
    )
    .sort_values(["domain"])
)

chunk_summary


,domain,chunk_count,avg_words,min_words,max_words
0,ecommerce,10,32.4,11,38
1,healthcare,10,33.2,11,38
2,software_docs,10,32.6,16,38


In [32]:
sample_chunks = (
    chunks_df[
        ["domain", "title", "chunk_index", "word_count", "chunk_text"]
    ]
    .query("title in ['Fasting Blood Test Preparation', 'Returns and Refunds Policy', 'Login Loop Troubleshooting']")
    .sort_values(["title", "chunk_index"])
    .reset_index(drop=True)
)

sample_chunks


,domain,title,chunk_index,word_count,chunk_text
0,healthcare,Fasting Blood Test Preparation,0,38,"For a fasting blood test, stop eating 8 hours before your appointment. Plain water is allowed and is encouraged unless your clinician gave a different instruction. Avoid coffee..."
1,healthcare,Fasting Blood Test Preparation,1,38,"tea, juice, and sports drinks during the fasting window because they can change glucose and lipid readings. Take regular medicines only if your clinician told you they are safe..."
2,healthcare,Fasting Blood Test Preparation,2,11,safe before the test. Bring your medication list to the appointment.
3,software_docs,Login Loop Troubleshooting,0,38,"If the app returns to the login screen immediately after authentication, first confirm that the system clock is correct and the browser can store secure cookies. On Windows, co..."
4,software_docs,Login Loop Troubleshooting,1,38,corporate endpoint tools may block loopback redirects. Add the desktop helper to the local allow-list before retrying login. If the browser opens but the token never returns to...
5,software_docs,Login Loop Troubleshooting,2,16,"the app, clear cached cookies for the app domain and retry with the latest desktop helper."
6,ecommerce,Returns and Refunds Policy,0,38,"Most unopened products can be returned within 30 days of delivery for a full refund if they include all original accessories and packaging. Opened hygiene products, activated s..."
7,ecommerce,Returns and Refunds Policy,1,37,"keys, and opened power accessories for laptops are not eligible for standard return unless they arrived damaged or defective. Refunds are issued to the original payment method ..."


# Section 6 — Retrieval Helpers

We will use three retrieval styles:

- TF-IDF
- lightweight semantic retrieval using TF-IDF + SVD
- hybrid retrieval

The purpose is to retrieve good chunk-level evidence from the same shared chunk table built with the Lab 8 chunking style.


In [33]:
def normalize_lexical_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_index_bundle(chunks_subset):
    lexical_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    lexical_matrix = lexical_vectorizer.fit_transform(
        chunks_subset["search_text"].map(normalize_lexical_text)
    )

    semantic_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    semantic_source = semantic_vectorizer.fit_transform(
        chunks_subset["search_text"].map(normalize_lexical_text)
    )

    max_components = min(
        12,
        max(1, semantic_source.shape[0] - 1),
        max(1, semantic_source.shape[1] - 1),
    )

    if max_components >= 2:
        semantic_svd = TruncatedSVD(n_components=max_components, random_state=42)
        semantic_matrix = normalize(semantic_svd.fit_transform(semantic_source))
    else:
        semantic_svd = None
        semantic_matrix = normalize(semantic_source.toarray())

    return {
        "lexical_vectorizer": lexical_vectorizer,
        "lexical_matrix": lexical_matrix,
        "semantic_vectorizer": semantic_vectorizer,
        "semantic_svd": semantic_svd,
        "semantic_matrix": semantic_matrix,
    }


def min_max_normalize(scores):
    scores = np.array(scores, dtype=float)
    if len(scores) == 0:
        return scores
    score_min = scores.min()
    score_max = scores.max()
    if score_max == score_min:
        return np.zeros_like(scores)
    return (scores - score_min) / (score_max - score_min)


def get_filtered_chunks(query_row):
    subset = chunks_df[chunks_df["domain"] == query_row["domain"]].copy()

    for field in ["platform", "version", "region", "audience"]:
        value = query_row.get(field)
        if pd.isna(value) or value is None:
            continue
        if field not in subset.columns:
            continue
        match_mask = subset[field].fillna("").str.contains(str(value), case=False, regex=False)
        if match_mask.any():
            subset = subset[match_mask].copy()

    return subset.reset_index(drop=True)


def retrieve_top_k(query_row, retriever="hybrid", k=5, alpha=0.45):
    subset = get_filtered_chunks(query_row)
    bundle = build_index_bundle(subset)

    lexical_query = bundle["lexical_vectorizer"].transform(
        [normalize_lexical_text(query_row["query"])]
    )
    lexical_scores = cosine_similarity(lexical_query, bundle["lexical_matrix"]).flatten()

    semantic_query = bundle["semantic_vectorizer"].transform(
        [normalize_lexical_text(query_row["query"])]
    )
    if bundle["semantic_svd"] is not None:
        semantic_query = normalize(bundle["semantic_svd"].transform(semantic_query))
    else:
        semantic_query = normalize(semantic_query.toarray())
    semantic_scores = cosine_similarity(semantic_query, bundle["semantic_matrix"]).flatten()

    lexical_norm = min_max_normalize(lexical_scores)
    semantic_norm = min_max_normalize(semantic_scores)
    hybrid_scores = alpha * lexical_norm + (1 - alpha) * semantic_norm

    score_map = {
        "tfidf": lexical_scores,
        "semantic": semantic_scores,
        "hybrid": hybrid_scores,
    }

    chosen_scores = score_map[retriever]
    ranking = np.argsort(chosen_scores)[::-1][:k]

    results = subset.iloc[ranking].copy()
    results["tfidf_score"] = lexical_scores[ranking]
    results["semantic_score"] = semantic_scores[ranking]
    results["score"] = chosen_scores[ranking]
    results["retriever"] = retriever
    return results.reset_index(drop=True)


In [34]:
sample_query = queries_df.loc[queries_df["query_id"] == "E1"].iloc[0]
retrieve_top_k(sample_query, retriever="hybrid", k=4)[
    ["title", "score", "chunk_text"]
]


,title,score,chunk_text
0,Returns and Refunds Policy,1.000000,"Most unopened products can be returned within 30 days of delivery for a full refund if they include all original accessories and packaging. Opened hygiene products, activated s..."
1,Returns and Refunds Policy,0.534598,"keys, and opened power accessories for laptops are not eligible for standard return unless they arrived damaged or defective. Refunds are issued to the original payment method ..."
2,Damaged Delivery Support,0.371291,"If an order arrives damaged, photograph the item, the shipping box, and the label before discarding the packaging. Submit the damage report through the order page within 48 hou..."
3,Laptop Warranty Guide,0.365416,"unless a product page states a shorter promotional warranty. Accidental damage, liquid exposure, and unauthorized repair are excluded from warranty coverage."


# Section 7 — Evaluate Retrieval on the Shared Chunk Table

Now we hold the chunk table constant and inspect how well retrieval returns useful evidence for the same queries.


In [35]:
def hit_rate_at_k(relevant_document_ids, returned_document_ids):
    return int(any(doc_id in relevant_document_ids for doc_id in returned_document_ids))


def precision_at_k(relevant_document_ids, returned_document_ids):
    if not returned_document_ids:
        return 0.0
    return sum(doc_id in relevant_document_ids for doc_id in returned_document_ids) / len(returned_document_ids)


def evaluate_query(query_id, retriever="hybrid", k=3):
    query_row = queries_df.loc[queries_df["query_id"] == query_id].iloc[0]
    results = retrieve_top_k(query_row, retriever=retriever, k=k)
    returned_document_ids = results["document_id"].tolist()

    return pd.DataFrame(
        [
            {
                "query_id": query_id,
                "domain": query_row["domain"],
                "query_type": query_row["query_type"],
                "hit_rate_at_k": hit_rate_at_k(query_row["relevant_document_ids"], returned_document_ids),
                "precision_at_k": round(
                    precision_at_k(query_row["relevant_document_ids"], returned_document_ids),
                    3,
                ),
                "top_titles": results["title"].tolist(),
            }
        ]
    )


evaluate_query("S2", retriever="hybrid", k=3)


,query_id,domain,query_type,hit_rate_at_k,precision_at_k,top_titles
0,S2,software_docs,troubleshooting,1,1.0,"[Login Loop Troubleshooting, Login Loop Troubleshooting, Login Loop Troubleshooting]"


In [36]:
evaluation_rows = []
for query_id in queries_df["query_id"]:
    evaluation_rows.append(evaluate_query(query_id, retriever="hybrid", k=3))

evaluation_results_df = pd.concat(evaluation_rows, ignore_index=True)

evaluation_summary = (
    evaluation_results_df.groupby(["domain", "query_type"], as_index=False)
    .agg(
        avg_hit_rate=("hit_rate_at_k", "mean"),
        avg_precision=("precision_at_k", "mean"),
    )
    .sort_values(["domain", "query_type"])
)

evaluation_summary


,domain,query_type,avg_hit_rate,avg_precision
0,ecommerce,factual,1.0,0.8335
1,ecommerce,policy,1.0,0.6670
2,ecommerce,troubleshooting,1.0,0.6670
3,healthcare,factual,1.0,1.0000
4,healthcare,policy,1.0,0.8335
5,healthcare,troubleshooting,1.0,0.3330
6,software_docs,factual,1.0,0.6670
7,software_docs,policy,1.0,0.3330
8,software_docs,procedure,1.0,0.6670
9,software_docs,troubleshooting,1.0,1.0000


## Interpreting the Results

Typical patterns you should expect:

- some queries retrieve the correct document immediately,
- some queries retrieve a partly useful chunk but still miss an important detail,
- metadata such as current/outdated, platform, region, or audience can improve the final evidence set,
- retrieval quality still shapes answer quality before the LLM is ever called.


# Section 8 — Build the Final Context Package

Candidate evidence is not final context.

The context package should:

- prefer current documents,
- keep the most useful evidence first,
- remove duplicates,
- stay inside a word budget,
- preserve metadata the LLM can cite later.


In [37]:
query_overview = queries_df[["query_id", "domain", "query_type", "query"]].copy()
query_overview


,query_id,domain,query_type,query
0,H1,healthcare,factual,Can I drink water before a fasting blood test?
1,H2,healthcare,troubleshooting,What should I do if I have chest pain and trouble breathing?
2,H3,healthcare,policy,Do I have to finish the antibiotic course if I feel better after two days?
3,H4,healthcare,policy,Can a 12-year-old come alone to a clinic appointment?
4,E1,ecommerce,policy,Can I return an opened laptop charger?
5,E2,ecommerce,troubleshooting,My order arrived with a cracked screen. What should I do first?
6,E3,ecommerce,factual,How long does normal shipping take in the UAE?
7,E4,ecommerce,factual,Does the laptop battery come with a two-year warranty?
8,S1,software_docs,factual,Which Python version is required for Desktop App 5.2?
9,S2,software_docs,troubleshooting,The app keeps sending me back to login on Windows. What should I check first?


## Context Packaging Rules

We do not send every retrieved chunk to the LLM.

We package only the most useful evidence so the final prompt stays focused and grounded.


In [38]:
def build_context_package(query_id, retriever="hybrid", k=6, word_budget=150, max_chunks=3):
    query_row = queries_df.loc[queries_df["query_id"] == query_id].iloc[0]

    candidates = retrieve_top_k(query_row, retriever=retriever, k=k).copy()
    candidates = candidates.sort_values(
        by=["is_current", "score", "effective_date"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    selected_rows = []
    seen_texts = set()
    used_words = 0

    for _, row in candidates.iterrows():
        normalized_text = re.sub(r"\s+", " ", row["chunk_text"]).strip().lower()
        if normalized_text in seen_texts:
            continue

        chunk_words = len(row["chunk_text"].split())
        if selected_rows and used_words + chunk_words > word_budget:
            continue

        selected_rows.append(row.to_dict())
        seen_texts.add(normalized_text)
        used_words += chunk_words

        if len(selected_rows) >= max_chunks:
            break

    selected_df = pd.DataFrame(selected_rows)
    context_blocks = []

    for idx, row in enumerate(selected_rows, start=1):
        current_label = "CURRENT" if row["is_current"] else "OUTDATED"
        context_blocks.append(
            f"[Source {idx}] {row['title']} | domain={row['domain']} | date={row['effective_date']} | {current_label}\n"
            f"{row['chunk_text']}"
        )

    return {
        "query_row": query_row,
        "candidates": candidates,
        "selected_chunks_df": selected_df,
        "used_words": used_words,
        "context_text": "\n\n".join(context_blocks),
    }


context_package_demo = build_context_package("S2")
context_package_demo["selected_chunks_df"][["title", "score", "chunk_text"]]


,title,score,chunk_text
0,Login Loop Troubleshooting,0.918788,"If the app returns to the login screen immediately after authentication, first confirm that the system clock is correct and the browser can store secure cookies. On Windows, co..."
1,Login Loop Troubleshooting,0.915399,corporate endpoint tools may block loopback redirects. Add the desktop helper to the local allow-list before retrying login. If the browser opens but the token never returns to...
2,Login Loop Troubleshooting,0.435783,"the app, clear cached cookies for the app domain and retry with the latest desktop helper."


In [39]:
package_summary = pd.DataFrame(
    [
        {
            "query_id": "S2",
            "used_words": context_package_demo["used_words"],
            "titles": context_package_demo["selected_chunks_df"]["title"].tolist(),
        }
    ]
)

package_summary


,query_id,used_words,titles
0,S2,92,"[Login Loop Troubleshooting, Login Loop Troubleshooting, Login Loop Troubleshooting]"


## Current Versus Outdated Conflicts

A chunk can be relevant and still be unsafe.

The classic example is a current guide versus an archived notice that still shares important keywords.


In [40]:
fasting_package = build_context_package("H1")
fasting_package["candidates"][["title", "is_current", "score", "chunk_text"]]


,title,is_current,score,chunk_text
0,Fasting Blood Test Preparation,True,1.000000,"For a fasting blood test, stop eating 8 hours before your appointment. Plain water is allowed and is encouraged unless your clinician gave a different instruction. Avoid coffee..."
1,Fasting Blood Test Preparation,True,0.887827,safe before the test. Bring your medication list to the appointment.
2,Fasting Blood Test Preparation,True,0.879605,"tea, juice, and sports drinks during the fasting window because they can change glucose and lipid readings. Take regular medicines only if your clinician told you they are safe..."
3,Antibiotic Course Instructions,True,0.004839,"missed, take it when remembered unless the next dose is due soon. Do not double the dose. Seek urgent review if rash, facial swelling, wheezing, or severe diarrhea develops whi..."
4,Urgent Care Triage Guide,True,0.004310,"Emergency signs: chest pain with breathing difficulty, sudden confusion, blue lips, or loss of consciousness require emergency services immediately. Same-day urgent clinic care..."
5,Old Fasting Advisory,False,0.727176,Legacy fasting advice from 2024 stated that patients should avoid both food and water after midnight before morning blood work. This notice remains archived for audit reasons a...


# Section 9 — The First LLM Connection

This is the first time we connect an LLM in the course sequence.

Important rule:

- the LLM is only the **final answer generator**
- retrieval, chunking, and context packaging happen before this point
- in this notebook we use **Ollama on the local device** rather than a cloud API


In [41]:
def build_grounded_prompt(query, context_text):
    return f"""You are a careful grounded assistant.

Use only the provided context.
If the context is insufficient, say so clearly.
If sources conflict, prefer CURRENT sources and mention the conflict.
Answer briefly but completely.

Question:
{query}

Context:
{context_text}
"""

In [47]:
import requests

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "deepseek-r1:1.5b")

print(f"OLLAMA_HOST={OLLAMA_HOST}")
print(f"OLLAMA_MODEL={OLLAMA_MODEL}")

try:
    tags_response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
    tags_response.raise_for_status()
    available_models = [model["name"] for model in tags_response.json().get("models", [])]
    print("Available Ollama models:", available_models if available_models else "No local models reported")
    if OLLAMA_MODEL not in available_models:
        print("Selected model is not listed. Pull it first with: ollama pull", OLLAMA_MODEL)
except Exception as exc:
    print("Could not reach the Ollama server. Start Ollama first, then rerun this cell.")
    print("Connection error:", exc)

OLLAMA_HOST=http://127.0.0.1:11434
OLLAMA_MODEL=deepseek-r1:1.5b
Available Ollama models: ['nomic-embed-text:latest', 'mistral:latest', 'deepseek-r1:1.5b', 'deepseek-r1:7b']


In [48]:
demo_prompt = build_grounded_prompt(
    context_package_demo["query_row"]["query"],
    context_package_demo["context_text"],
)

print("=== GROUNDED PROMPT PREVIEW ===")
print(demo_prompt[:1600])


=== GROUNDED PROMPT PREVIEW ===
You are a careful grounded assistant.

Use only the provided context.
If the context is insufficient, say so clearly.
If sources conflict, prefer CURRENT sources and mention the conflict.
Answer briefly but completely.

Question:
The app keeps sending me back to login on Windows. What should I check first?

Context:
[Source 1] Login Loop Troubleshooting | domain=software_docs | date=2026-03-14 | CURRENT
If the app returns to the login screen immediately after authentication, first confirm that the system clock is correct and the browser can store secure cookies. On Windows, corporate endpoint tools may block loopback redirects. Add the desktop

[Source 2] Login Loop Troubleshooting | domain=software_docs | date=2026-03-14 | CURRENT
corporate endpoint tools may block loopback redirects. Add the desktop helper to the local allow-list before retrying login. If the browser opens but the token never returns to the app, clear cached cookies for the app domain 

In [50]:
def ask_ollama(prompt, model=None, temperature=0):
    model = model or OLLAMA_MODEL

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }

    try:
        response = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=120)
        response.raise_for_status()
    except Exception as exc:
        return f"Ollama request failed: {exc}"

    data = response.json()
    return data.get("response", "No response field returned by Ollama.")


demo_answer = ask_ollama(demo_prompt)
print("DEMO ANSWER:\n", demo_answer)


DEMO ANSWER:
 <think>
Okay, so I'm trying to figure out why my app is sending me back to the login screen on Windows. The user mentioned that they're using a login loop, which probably means their app is trying to log in from another source but isn't working correctly.

Looking at the context provided, there are three sources listed. Each of these seems to address different issues related to loopback redirects and cookies. But I'm not entirely sure how all of them connect to the problem at hand.

First, Source 1 says that if the app returns immediately after authentication, they should check if the system clock is correct and if the browser can store secure cookies. On Windows, corporate endpoint tools might block loopback redirects, so adding a desktop helper could help. It also mentions clearing cached cookies for the app domain if the token doesn't return.

Source 2 seems to provide similar advice but from a different angle. It suggests using corporate endpoint tools and adding a de

In [51]:
def extract_final_answer(raw_text):
    if not raw_text:
        return ""                                                                                                                                                                                                                                                     # Remove hidden reasoning blocks like <think> ... </think>
    cleaned = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL | re.IGNORECASE)

    # Clean extra blank lines
    cleaned = re.sub(r"\n\s*\n+", "\n\n", cleaned).strip()
    return cleaned

final_answer = extract_final_answer(demo_answer)
print(final_answer)

The app is likely sending you back to login due to issues with the system clock and secure cookies on Windows. First, ensure your system clock is correct and that the browser can store secure cookies. If these are not working, consider using corporate endpoint tools and adding a desktop helper before retrying login. Additionally, clearing cached cookies for the app domain might resolve the issue if the token doesn't return. Try these steps in order to diagnose and fix the problem.


## What You Should Observe

When this final-answer stage is prepared correctly, the teaching point should be obvious:

- the LLM answer quality still depends on evidence quality,
- a cleaner context package produces a more grounded answer,
- the model should stay inside the retrieved evidence rather than invent missing details,
- local Ollama generation still depends on retrieval quality more than model magic.


# Section 10 — Failure Analysis

Every bad answer does not come from the same layer.

We classify failures into:

- chunking failure
- retrieval failure
- context assembly failure
- generation failure


In [45]:
failure_cases = pd.DataFrame(
    [
        {
            "case": "Fasting water conflict",
            "likely_failure_layer": "context assembly",
            "what_went_wrong": "An outdated fasting advisory can still look relevant if current-vs-outdated filtering is weak.",
            "better_fix": "Prefer CURRENT sources and keep the current preparation chunk ahead of the archived notice.",
        },
        {
            "case": "Opened laptop charger return",
            "likely_failure_layer": "chunking",
            "what_went_wrong": "A weak chunk boundary can separate the general return rule from the opened-accessory exception.",
            "better_fix": "Inspect the retrieved chunks and adjust chunk size or overlap so answerable evidence stays together.",
        },
        {
            "case": "Windows login loop",
            "likely_failure_layer": "retrieval",
            "what_went_wrong": "A broad query may retrieve generic login text before the Windows-specific fix.",
            "better_fix": "Use metadata-aware filtering and inspect whether the top chunks actually match the user's platform.",
        },
        {
            "case": "Postgres backend switch",
            "likely_failure_layer": "generation",
            "what_went_wrong": "Even good chunks can produce a vague answer if the final prompt does not force evidence-only response behavior.",
            "better_fix": "Use a grounded prompt with an evidence boundary and insufficient-information rule.",
        },
    ]
)

failure_cases


,case,likely_failure_layer,what_went_wrong,better_fix
0,Fasting water conflict,context assembly,An outdated fasting advisory can still look relevant if current-vs-outdated filtering is weak.,Prefer CURRENT sources and keep the current preparation chunk ahead of the archived notice.
1,Opened laptop charger return,chunking,A weak chunk boundary can separate the general return rule from the opened-accessory exception.,Inspect the retrieved chunks and adjust chunk size or overlap so answerable evidence stays together.
2,Windows login loop,retrieval,A broad query may retrieve generic login text before the Windows-specific fix.,Use metadata-aware filtering and inspect whether the top chunks actually match the user's platform.
3,Postgres backend switch,generation,Even good chunks can produce a vague answer if the final prompt does not force evidence-only response behavior.,Use a grounded prompt with an evidence boundary and insufficient-information rule.


In [46]:
assignment_query_overview = queries_df[["query_id", "domain", "query_type", "query"]].copy()
assignment_query_overview


,query_id,domain,query_type,query
0,H1,healthcare,factual,Can I drink water before a fasting blood test?
1,H2,healthcare,troubleshooting,What should I do if I have chest pain and trouble breathing?
2,H3,healthcare,policy,Do I have to finish the antibiotic course if I feel better after two days?
3,H4,healthcare,policy,Can a 12-year-old come alone to a clinic appointment?
4,E1,ecommerce,policy,Can I return an opened laptop charger?
5,E2,ecommerce,troubleshooting,My order arrived with a cracked screen. What should I do first?
6,E3,ecommerce,factual,How long does normal shipping take in the UAE?
7,E4,ecommerce,factual,Does the laptop battery come with a two-year warranty?
8,S1,software_docs,factual,Which Python version is required for Desktop App 5.2?
9,S2,software_docs,troubleshooting,The app keeps sending me back to login on Windows. What should I check first?


# Section 11 — Final Assignment

Use the provided corpus and notebook helpers to complete the following tasks.


## Task 1 — Inspect the Corpus and Fixed Chunk Table

For each domain:

- review at least one full source document,
- inspect the generated fixed-size chunks for that document,
- explain what information stays together or gets split across chunks.

## Task 2 — Evaluate Retrieval Quality

For at least two queries per domain:

- run retrieval on the shared chunk table,
- report Hit Rate@3 and Precision@3,
- explain which returned chunks were actually answerable.

## Task 3 — Build Context Packages

For at least one query in each domain:

- show the raw retrieved candidates,
- show the final context package,
- explain every important keep/drop decision.

## Task 4 — Run the First LLM Answer

For at least one query per domain:

- build the final grounded prompt,
- run the answer cell,
- explain how retrieval and context quality affected the final answer.

## Task 5 — Analyze Failure Cases

Choose at least three failures and classify them as:

- chunking failure
- retrieval failure
- context assembly failure
- generation failure


# Final Takeaways

- chunking is still necessary even when you use only one chunking path
- retrieved chunks are candidate evidence, not final context
- metadata can improve which evidence survives into the final package
- the LLM is the final step, not the whole system

If the answer is bad, do not blame the LLM first.
Check what evidence it was allowed to see.
